<a href="https://colab.research.google.com/github/edwardoughton/Agentic-GeoAI/blob/main/03_01_ggs662_agents_react.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3: From prompts to agents — ReAct and Agentic GeoAI

This lesson turns the conceptual design from Assignment 1 into an inspectable agent workflow. We first establish what an agent is (and is not), then work through **ReAct**: an iterative pattern in which a model reasons, takes a tool action, observes the result, and updates its next step.

The central principle remains: an agent may automate work, but it does not automate scientific responsibility.

## Learning objectives

By the end of class, you should be able to:

- distinguish an LLM, a prompt, a workflow, and an AI agent;
- describe the perception–reasoning/planning–action–observation loop;
- read and critique a ReAct trace;
- identify tools, state, stopping conditions, guardrails, and test oracles for a GeoAI agent;
- explain why geospatial work needs domain and spatial validation beyond code execution; and
- start and supervise an agent session in the VS Code Agents window.

## Where Week 2 and Assignment 1 left us

Week 2 treated the agent as a bounded implementer: the human specified the objective, unit of analysis, inputs, operations, missing-data rule, claim boundary, and validation checks. Assignment 1 asks you to propose those same components for a meaningful geographic problem.

Today we add the missing execution model. An agent does not simply produce a one-shot answer. It can inspect an environment, choose a permitted action, read the resulting observation, and revise its next action. This is useful only when the loop is **bounded, observable, and checked**.

## Essential vocabulary recap

| Term | Working definition for this course | A geospatial example |
|---|---|---|
| **LLM** | A model that predicts/generates text or other tokens from context. It has no direct access to your files or map unless a system gives it that access. | Explains what a coordinate reference system (CRS) is. |
| **Prompt** | Instructions and context given to a model. | “Explain the difference between hazard and risk.” |
| **Tool** | A callable capability outside the model: a function, shell command, GIS library, database, API, or browser. | Read a GeoPackage, reproject a layer, query a catalog. |
| **Agent** | A system that uses a model to pursue a goal through an iterative loop of context/perception, decision, tool action, and observation. | Inspects a dataset, writes a script, runs tests, then reports evidence. |
| **Agentic** | Describes a workflow that gives a model some delegated initiative to select or sequence actions toward a goal. It does **not** imply reliable autonomy. | Chooses whether to inspect columns before proposing a buffer operation. |
| **Environment/state** | The current world the agent can affect or observe: files, data, tool outputs, messages, and permissions. | Current working directory, layer schema, CRS, test results. |
| **Observation** | Tool output fed back into the agent’s working context. | `EPSG:4326`, a failed assertion, or a map preview. |
| **Guardrail** | A constraint that limits actions or claims. | Read-only inputs; no package installs; do not call an exposure count “risk.” |
| **Test oracle** | An independently justified expected result used to judge output. | A known facility must have exposure count 2. |
| **Human in the loop** | A person retains authority over consequential decisions and reviews evidence. | Chooses the risk definition and decides whether a result supports a claim. |

## A useful distinction: chat, workflow, and agent

A chat response may be useful, but it is not automatically an agent.

```text
Prompted LLM:  question → model response
Fixed workflow: input → predefined steps → output
Agent:        goal + state → decide → tool action → observation ┐
                              ↑___________________________________┘
```

The boundary is not whether the system uses an LLM. It is whether it can take actions in an environment and use the observations to choose what to do next. A fixed GIS script is often preferable when every step is already known; adding an agent adds cost, nondeterminism, and a larger failure surface.

## The agent loop

For a bounded GeoAI task, use this mental model:

```text
Human specification ──► Agent context/state
                             │
                   reason / plan / choose
                             │
                     permitted tool action
                             │
                   observation / test result
                             │
                    validate or iterate ──► stop + evidence
```

A good specification adds: (1) allowed tools and files, (2) success criteria, (3) a maximum number of iterations or cost budget, and (4) clear escalation conditions. “Keep trying until it works” is not a scientific stopping condition.

## ReAct: reasoning + acting

Yao et al. (2023) introduced **ReAct** as a prompting pattern that interleaves reasoning traces with actions and observations. The model’s intermediate text helps it track a plan; tool use lets it obtain information instead of relying only on its parameters. The pattern is often represented as:

```text
Thought:  What do I need to establish next?
Action:   Call one permitted tool with explicit arguments.
Observation: Record exactly what the tool returned.
... repeat until an acceptance criterion is met ...
Final:    State the result, its evidence, and its limitations.
```

In production interfaces, we should not require hidden model reasoning to be revealed. What matters for audit is an **action-and-evidence record**: tool calls, inputs, outputs, checks, edits, and the basis for the final claim. Treat a narrated rationale as a clue, not proof.

## A miniature ReAct trace: selecting a layer safely

**Goal:** Prepare a road layer for a 500 m hospital-access screen. The project specification says analysis must use a projected CRS in metres, preserve the source file, and stop if the CRS is missing.

| Step | Agent record | Why it is (or is not) adequate |
|---|---|---|
| 1 | Decision: inspect the layer metadata before buffering. | Good: unit-dependent GIS operation requires a CRS check. |
| 2 | Action: `inspect_vector(path='data/roads.geojson')` | A narrow, read-only action. |
| 3 | Observation: 4,210 features; CRS = EPSG:4326; geometry = LineString. | Critical: degrees are not metres. |
| 4 | Decision: reproject a copy to a locally appropriate projected CRS; record it in provenance. | Good only if the human/specification has authorized a CRS choice. |
| 5 | Action: `reproject(input='...', output='work/roads_projected.gpkg', crs='EPSG:26918')` | Writes a new artifact; source remains untouched. |
| 6 | Observation: output CRS = EPSG:26918; 4,210 features retained. | Necessary, but not sufficient: geometry and spatial plausibility should still be checked. |
| 7 | Action: `validate_bounds(...)` and `run_known_case(...)` | Tests the transformation and the analysis rule. |
| 8 | Final: report output path, CRS, checks passed, and the boundary: this is access screening, not travel time or health equity. | A bounded, evidence-oriented conclusion. |

Notice that the tool action is not the proof. The checks and the claim boundary are part of the work.

### Exercise 1 — audit the trace

With a partner, identify:

1. One decision that should be made by the human before the agent starts.
2. Two observations the agent needs before claiming the buffer is correct.
3. One plausible way the workflow could run without an error but still be wrong.
4. One stopping condition that is stronger than “the map looks right.”

## Why spatial agents need extra care

An agent can produce syntactically valid code and an attractive map while the analysis remains invalid. Spatial workflows add several recurring risks:

| Risk | Example | Check or guardrail |
|---|---|---|
| CRS and units | Buffering EPSG:4326 geometries by `500` interprets degrees, not metres. | Assert CRS and units before distance/area operations. |
| Spatial joins | A point on a boundary matches zero or multiple polygons. | State predicate (`within`, `intersects`, nearest), inspect unmatched/multiple matches. |
| Scale / MAUP | County averages conceal neighborhood variation. | State unit, resolution, aggregation, and sensitivity checks. |
| Temporal mismatch | 2020 population with 2026 flood extent. | Record timestamps and decide whether the combination is defensible. |
| Provenance / licensing | Tool discovers an undocumented data download. | Pin sources/versions and review license, authority, and completeness. |
| Semantic overclaim | Exposure index is described as risk. | Encode claim boundaries in prompt, outputs, and review checklist. |
| Visual persuasion | A choropleth’s classification makes a weak pattern look strong. | Inspect raw values, legend, classification, missingness, and uncertainty. |

## ReAct is a pattern, not a guarantee

ReAct can make a trajectory easier to inspect and can ground some steps in tool observations. It does **not** ensure that:

- the agent chose the right tool or interpreted its output correctly;
- the data are valid, current, representative, or ethically appropriate;
- a passing test suite covers the relevant scientific claim;
- the model did not invent a rationale, tool result, citation, or spatial interpretation; or
- a result is fair, causal, or decision-ready.

The practical response is not “trust the trace.” It is to predefine validation, keep permissions narrow, retain provenance, and review the result at the level of the decision being supported.

## VS Code Agents window: a supervised workspace

VS Code’s **Agents window** is a separate, agent-first surface for starting and monitoring agent sessions. It shares the workspace/worktree with the main VS Code window, so the same project files can be inspected in the editor and reviewed through Git. Its exact providers, models, and capabilities depend on your installed VS Code version, extensions, account, and institutional access.

Open it with the Command Palette: **`Ctrl+Shift+P` → `Chat: Open Agents window`**. Keep your project folder open in the main VS Code window. Start a new session, select an available agent/provider, and give the session a bounded task. The agent may propose edits and run tools according to its permissions; you remain responsible for reviewing changes and terminal activity.

The Agents window is a useful interface for observing the agent loop, not a replacement for problem formulation or peer review. See the official [VS Code Agents window documentation](https://code.visualstudio.com/docs/agents/run/agents-window).

### A safe first Agents-window routine

1. Open the course repository folder, then run `git status` in a VS Code terminal. Do not begin with unrelated uncommitted work.
2. Open the Agents window and start a new session. Give it one small, reversible, inspectable task.
3. State scope, allowed files/tools, outputs, validation, and a stopping condition. Explicitly rule out browsing, installations, or source-file changes if those are not needed.
4. Read its plan and every proposed change. Watch terminal commands; deny or pause actions outside the task.
5. Run independent checks yourself. Inspect `git diff` before accepting work; preserve the prompt, output, and evidence in your project notes.

For today, use a scratch directory or a new Git branch if you will allow edits. Never paste secrets (API keys, passwords, restricted data) into an agent chat.

### Exercise 2 — observe a bounded agent task

In a small scratch folder, ask an available agent to do the following. Adapt interface wording if your installation uses a different provider.

> Work only in this scratch folder. First inspect the existing files and give a plan of at most three bullets. Create `README.md` containing: (1) the current date as a placeholder `YYYY-MM-DD`, (2) a one-sentence description of a hypothetical spatial dataset, and (3) three validation questions. Do not browse, install packages, change existing files, or run commands other than listing files and showing the created file. Stop after reporting the file created.

Record: What was the agent’s plan? Which tool actions did it take? Which action was read-only and which changed state? Did the final report match the actual file? Delete the scratch artifact yourself after class if desired.

## From Assignment 1 concept to agent specification

Assignment 1 asks for objective, user context, inputs, workflow, tools, human–agent responsibilities, outputs, validation, failure modes, and scope/cost. Turn that into an executable but bounded specification:

| Component | Question to make explicit |
|---|---|
| Goal | What narrow artifact should exist at the end? |
| State and inputs | Which exact files, data versions, and metadata may the agent read? |
| Tools | What may it call? What must it not call? |
| Policy | Which interpretations and thresholds are fixed by the researcher? |
| Outputs | Which tables, map(s), code, logs, and documentation are required? |
| Validation | Which schema, spatial, statistical, and known-case checks must pass? |
| Claim boundary | What conclusion is explicitly unsupported? |
| Budget / stopping | How many turns, tools, files, or tokens; when must it stop or escalate? |

### Exercise 3 — write a ReAct-ready micro-specification

Choose one small step from your Assignment 1 proposal—not the whole final system. In 150 words or fewer, write:

- one goal and one non-goal;
- the exact input(s) and allowed tool(s);
- two actions the agent might take and the observations it needs after each;
- one human decision it may not make;
- two validation checks, including one known or hand-calculated case; and
- a stopping or escalation condition.

Exchange with a partner. Can they identify an action the agent could take outside the intended scope? If so, add a guardrail.

## A toy, deterministic ReAct-style loop

The next cell is deliberately **not** an LLM agent. It makes the control loop visible without an API key, cost, or hidden model behavior. Read the trace: each action produces an observation; the loop stops only when its explicit checks pass.

In [ ]:
# A tiny simulated environment: no packages or model calls needed.
dataset = {
    'name': 'hospital_access_screen',
    'crs': 'EPSG:4326',
    'features': 12,
    'source_preserved': True,
}
trace = []

def observe_metadata(data):
    return {'crs': data['crs'], 'features': data['features']}

def reproject_copy(data, target_crs):
    return {**data, 'crs': target_crs, 'derived_copy': True}

# Policy supplied by a researcher, not selected by the agent.
required_crs = 'EPSG:26918'
max_actions = 3

for _ in range(max_actions):
    if dataset['crs'] != required_crs:
        trace.append(('decision', 'Distance analysis needs projected metre units.'))
        trace.append(('action', 'inspect_metadata'))
        trace.append(('observation', observe_metadata(dataset)))
        trace.append(('action', f'reproject_copy -> {required_crs}'))
        dataset = reproject_copy(dataset, required_crs)
        trace.append(('observation', observe_metadata(dataset)))
    elif dataset['features'] > 0 and dataset['source_preserved']:
        trace.append(('validation', 'Projected CRS, non-empty output, source preserved: PASS'))
        break

for kind, record in trace:
    print(f'{kind.upper()}: {record}')
print('FINAL STATE:', dataset)

### Exercise 4 — extend the safety case

The toy loop passes its own checks. Is that enough to authorize a 500 m access analysis? Add one check that would expose each of these problems:

- EPSG:26918 is not appropriate for the study area;
- road geometry is invalid or empty after transformation;
- the 500 m threshold has no substantive justification;
- hospitals or roads represent different years.

Which of these can be automated, and which requires a human/domain judgment?

## Applications: where agents can help—and where they should not decide

| Potential application | Good delegated subtask | Human responsibility |
|---|---|---|
| Disaster data triage | Find documented files, profile schema, flag missing values, produce a reproducible extract. | Define emergency question, authoritative sources, acceptable uncertainty, and decision use. |
| EO / image workflow | Prepare a manifest, run a specified model, calculate agreed metrics, make QA figures. | Choose labels/model validity, interpret errors and generalization. |
| Accessibility analysis | Implement a stated network/threshold procedure and compare scenarios. | Define access, population groups, network assumptions, and equity interpretation. |
| Planning document review | Retrieve and organize passages with provenance. | Interpret policy, resolve ambiguity, and make normative recommendations. |
| Spatial coding support | Inspect a codebase, make a narrow edit, run tests, summarize a diff. | Approve design, validate scientific semantics, and merge work. |

The most credible near-term applications use agents to make bounded analysis more inspectable and reproducible—not to delegate high-stakes judgment.

## Reading list and references

### Foundations: agents, tools, and evaluation

- Russell, S. J. & Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.), Chs. 2–3. Rational agents and task environments.
- Yao, S. et al. (2023). [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629). ICLR. The core paper for today’s loop.
- Wei, J. et al. (2022). [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903). Distinguish reasoning prompts from actions in an external environment.
- Schick, T. et al. (2023). [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761). A foundational tool-use approach.
- Karpas, E. et al. (2022). [MRKL Systems: A modular, neuro-symbolic architecture](https://arxiv.org/abs/2205.00445). A useful framing for routing to specialized tools.
- Liu, X. et al. (2023). [AgentBench: Evaluating LLMs as Agents](https://arxiv.org/abs/2308.03688). Why multi-environment evaluation matters.

### GeoAI and geospatial agents

- Chen, Y., Wang, W., Lobry, S. & Kurtz, C. (2024). [GeoAgent: An LLM Agent for Automatic Geospatial Data Analysis](https://arxiv.org/abs/2410.18792). A direct example combining code execution, static analysis, retrieval, and geospatial task evaluation.
- Zaytar, A. et al. (2026). [GeoAI Agency Primitives](https://arxiv.org/abs/2604.01869). A proposal for human-centered GIS agent capabilities and evaluation. Preprint—read critically.
- Dorobantu, G. I. & Badea, A. C. (2026). [Geospatial reasoning and awareness in large language models: a systematic review](https://doi.org/10.1007/s10462-026-11512-x). Review of LLM geospatial capabilities and remaining autonomy limitations.
- Li, H. et al. (2024). [Mapping the landscape and roadmap of GeoAI in quantitative human geography](https://doi.org/10.1016/j.jag.2024.103734). Broad GeoAI context across human-geography applications.

### Interface documentation

- Microsoft. [Use the Agents window (Preview)](https://code.visualstudio.com/docs/agents/run/agents-window). Interface details change quickly; use this official page for the current setup and capabilities.

## Before next week

Revise your Assignment 1 presentation so its proposed agent has a visible execution loop: inputs/state → permitted tool action → observation → validation → stopping/escalation. Be ready to explain one workflow step using the terms **action**, **observation**, **guardrail**, and **test oracle**.

Keep your original intellectual contribution visible. An AI-use statement should name tools used and explain how you reviewed, tested, and revised any generated material. Do not use fabricated citations or submit code you cannot explain.